## Notebook Overview

### Micro-Level Job Data Preprocessing

This notebook processes the scraped job listing dataset and prepares it for analysis.

The raw dataset was collected via the Jobup.ch web scraping pipeline and contains
job listings for data-related roles such as:

- Data Scientist
- Data Analyst
- Machine Learning Engineer
- Data Engineer
- AI Engineer

Purpose of this notebook:

1. Validate and clean the raw dataset
2. Standardize text fields
3. Extract structured information from job listings
4. Reconstruct salary transparency indicators
5. Classify job seniority levels
6. Extract skills and compute skill counts
7. Produce a clean micro-level dataset for integration with macro datasets

Output dataset:

data/processed/jobs_micro_cleaned_final.csv

In [ ]:
# Environment Setup

import pandas as pd
import numpy as np
import re
from pathlib import Path

# Project paths
DATA_PROCESSED = Path("../data/processed")

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load Dataset

file_path = DATA_PROCESSED / "jobs_combined_2026-03-01T17-21-57.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

## Structural Validation

Before cleaning the dataset we perform basic quality checks:

• dataset dimensions  
• missing values  
• duplicate records  
• column structure

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Remove Duplicates

df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after duplicate removal:", df.shape)

## Text Cleaning

Text fields such as job titles and descriptions are cleaned to improve consistency
and enable later text analysis.

Cleaning steps:

• convert text to lowercase  
• remove unnecessary whitespace  
• remove line breaks and formatting artifacts

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return text

    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text

df["title_clean"] = df["title"].apply(clean_text)
df["description_clean"] = df["description"].apply(clean_text)

## Location Processing

The job location is standardized by extracting the city name and mapping it to
the corresponding Swiss canton.

In [ ]:
df["city_clean"] = df["city"].str.strip().str.title()

# Example canton mapping
canton_map = {
    "Zurich": "ZH",
    "Geneva": "GE",
    "Lausanne": "VD",
    "Bern": "BE",
    "Basel": "BS"
}

df["canton"] = df["city_clean"].map(canton_map)

## Salary Transparency

Many job listings do not explicitly state salaries.

A binary indicator `salary_available` is constructed based on
the presence of salary information in the job listing.

In [ ]:
df["salary_available"] = df["salary"].notna().astype(int)

## Seniority Classification

Job titles are classified into seniority levels using keyword matching.

Levels:

• intern  
• junior  
• mid  
• senior  
• lead

In [ ]:
def classify_seniority(title):

    title = str(title).lower()

    if "intern" in title:
        return "intern"

    if "junior" in title:
        return "junior"

    if "senior" in title:
        return "senior"

    if "lead" in title or "principal" in title:
        return "lead"

    return "mid"

df["seniority"] = df["title_clean"].apply(classify_seniority)

## Skill Extraction

Skills are extracted from job descriptions using keyword matching.

This provides an estimate of the technical complexity of job postings.

In [ ]:
skills = [
    "python",
    "sql",
    "r",
    "machine learning",
    "tensorflow",
    "pytorch",
    "pandas",
    "spark",
    "aws",
    "azure"
]

def extract_skills(text):

    found = []

    for skill in skills:
        if skill in str(text):
            found.append(skill)

    return found

df["skills"] = df["description_clean"].apply(extract_skills)
df["skill_count"] = df["skills"].apply(len)

## Final Dataset Construction

Only the variables required for analysis are kept in the final dataset.

In [ ]:
df_final = df[[
    "title_clean",
    "company",
    "city_clean",
    "canton",
    "seniority",
    "salary_available",
    "skill_count"
]]

In [ ]:
# Validation

print("Final dataset shape:", df_final.shape)

print("\nSeniority distribution:")
print(df_final["seniority"].value_counts())

print("\nSalary transparency rate:")
print(df_final["salary_available"].mean())

In [ ]:
# Export Clean Dataset

output_path = DATA_PROCESSED / "jobs_micro_cleaned_final.csv"

df_final.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)